# Trainable CGenFF LJ scales — a walkthrough

**Goal.** Replace a hand-tuned Lennard-Jones grid scan with σ/ε parameters that
are *learned* by gradient descent, alongside the ML potential, and then deploy
them in a condensed-phase MD run.

This notebook is meant to be read top to bottom. Cells 1–6 run in seconds on a
laptop CPU with no CHARMM and no GPU — they are the conceptual core. The
expensive steps (real dataset prep, real training, MD) are shown as commands to
run on the cluster, not executed inline.

**Companion material**

| What | Where |
|---|---|
| Reference page | `docs/hybrid-mm-lj-scales.md` |
| Cluster job (prep → train → MD) | `examples/hybrid_mm_charges/submit_lj_scales_scicore.sbatch` |
| Training config | `examples/hybrid_mm_charges/train_fixed_lj_scales.yaml` |
| MD config | `examples/hybrid_mm_charges/md_fixed_lj_scales.yaml` |
| Tests that pin all of this | `tests/unit/test_mm_lj_scales.py`, `tests/unit/test_mm_lj_scales_learning.py` |
| Tracking issue | [#133](https://github.com/EricBoittier/mmml/issues/133) |

## 1. What is actually being trained

CGenFF assigns every atom an **atom type** (`CG331`, `HGA3`, `CLGA1`, …). Each
type has a fixed σ (radius) and ε (well depth) in the CGenFF master tables. We do
**not** invent new types and we do **not** free the tables. Instead each type gets
one multiplicative scale, initialised at exactly 1.0:

$$\sigma^{\mathrm{eff}}_t = \sigma^{\mathrm{CGenFF}}_t \cdot s^{\sigma}_t
\qquad
\varepsilon^{\mathrm{eff}}_t = \varepsilon^{\mathrm{CGenFF}}_t \cdot s^{\varepsilon}_t$$

So a model with 2 relevant types learns 4 numbers. Starting at 1.0 means an
untrained model is *bit-identical* to stock CGenFF — a useful property, because
any change you see is attributable to training.

These scales live as two ordinary leaves on the Optax parameter tree, next to the
neural network weights, so the same optimizer updates both.

## 2. Why we train LJ *without* Ewald

This is the part that confuses people most, so it comes before any code.

You will read "LJ is forced off under `lr_solver: ewald`" and assume it is a
missing feature to work around. **It is not — training under truncated MIC is the
recommended path.** The reason is that the two nonbonded terms have genuinely
different long-range requirements:

| Term | Falls off as | Lattice sum | Consequence |
|---|---|---|---|
| LJ | $r^{-12}$, $r^{-6}$ | converges absolutely, fast | a switch at 8–13 Å is standard practice |
| Coulomb | $r^{-1}$ | only *conditionally* convergent | truncation is qualitatively wrong → Ewald/PME required |

σ and ε are short-ranged parameters, and we fit them with the same short-ranged
switched operator we later deploy. Nothing the Ewald sum would have told us about
a well depth is lost.

### The catch you must understand

Because Coulomb is truncated while you fit, **Coulomb error can be absorbed into
the LJ parameters**. This is real parameter compensation and it is the main way
this workflow goes wrong: σ/ε that fit beautifully and then behave badly under
Ewald, because part of what they learned was standing in for missing
electrostatics.

Three habits keep it bounded — do all three:

1. **Fit with `mm_charge_mode: fixed`.** Fixed CGenFF charges cannot co-adapt, so
   compensation has one fewer place to hide.
2. **Keep cutoffs identical between training and MD** (`mm_switch_on`,
   `mm_switch_width`, `ml_switch_width`). These *define* the operator your σ/ε
   were fitted against.
3. **Validate on something that was not in the loss** — density, or an RDF
   first-peak position. A fit that only reports its own training loss has not
   been validated.

## 3. Environment check

Nothing here needs a GPU or CHARMM.

In [ ]:
import json, sys
from pathlib import Path

import jax, jax.numpy as jnp
import numpy as np

# Repo root, whether the notebook is opened from examples/hybrid_mm_charges/ or the root.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(REPO))

print("repo        :", REPO)
print("jax         :", jax.__version__, "| devices:", jax.devices())
print("x64 enabled :", jax.config.jax_enable_x64)

## 4. The input data, and the PSF-ordering trap

Hybrid MM training needs a dimer NPZ carrying three extra per-atom fields on top
of the usual `R/Z/E/F`:

| Field | Meaning |
|---|---|
| `cgenff_type_idx` | row into the master σ/ε tables (`-1` = padding) |
| `cgenff_charge` | CGenFF partial charge |
| `mol_id` | which monomer each atom belongs to (`-1` = padding) |

Raw QM output does **not** have these — they are added by `mmml prepare-mm-dataset`.

**The trap:** atom ordering must match the CHARMM PSF, because CGenFF type
assignment walks the topology. Two files can hold identical data in different
orders and only one is usable. Compare the first frame's `Z`:

In [ ]:
candidates = {
    "dcm_mp2_psf_order.npz":            REPO / "examples/dcm_mp2_psf_order.npz",
    "new-dcm-round-2-only_MP2_41950.npz": REPO / "examples/new-dcm-round-2-only_MP2_41950.npz",
}
SYMBOL = {1: "H", 6: "C", 8: "O", 17: "Cl"}

for name, path in candidates.items():
    if not path.is_file():
        print(f"{name:38s} (not present on this machine)")
        continue
    d = np.load(path, allow_pickle=True)
    z = np.asarray(d["Z"][0])
    z = z[z > 0]                                  # strip padding
    order = " ".join(SYMBOL.get(int(v), str(v)) for v in z)
    has_cgenff = {"cgenff_type_idx", "cgenff_charge", "mol_id"} <= set(d.files)
    print(f"{name:38s} frame0 = {order:20s} cgenff_ready={has_cgenff}")

Both are the same MP2 DCM data, but only `dcm_mp2_psf_order.npz` is in PSF order
(**C Cl Cl H H**); the other is **C H H Cl Cl**. Feeding the wrong one does not
crash — it silently mis-assigns types, which is far worse. Neither has the CGenFF
fields yet: that is the next step.

### Running the assignment

This takes minutes on 40k frames, so run it on the cluster rather than here:

```bash
mmml prepare-mm-dataset \
  --data examples/dcm_mp2_psf_order.npz \
  --output artifacts/lj_scales_dcm/dcm_mp2_cgenff.npz \
  --num-workers 4
```

Then **assert** the output rather than trusting it — a silent prep failure that
reaches training wastes GPU hours:

```python
d = np.load("artifacts/lj_scales_dcm/dcm_mp2_cgenff.npz")
assert {"cgenff_type_idx", "cgenff_charge", "mol_id"} <= set(d.files)
```

Schema reference: `docs/hybrid-mm-dataset-preparation.md`.

## 5. The master tables and the parameter tree

Now the mechanics. `cgenff_type_names_from_prm()` returns type names in exactly
the same order as the master σ/ε arrays — that ordering is what makes it possible
to map a trained scale vector back onto CHARMM's atom-type list later.

In [ ]:
from mmml.models.mm_lj_scales import (
    attach_mm_lj_scales,
    split_mm_lj_scale_params,
    apply_mm_lj_scales,
)

# A tiny stand-in for the master tables (real ones have ~1200 CGenFF types).
# Values are in the range of real CG331 / HGA3 entries.
MASTER_SIGMAS   = jnp.array([3.6527, 2.3876])
MASTER_EPSILONS = jnp.array([0.0780, 0.0240])
TYPE_NAMES      = ["CG331", "HGA3"]

# `params` is the Optax pytree: network weights under "params", plus two leaves.
params = attach_mm_lj_scales({"params": {"...network weights..."}}, n_types=2)
print("leaves on the parameter tree:", sorted(params.keys()))
print("  sigma scale init  :", np.asarray(params["mm_lj_sigma_scale"]))
print("  epsilon scale init:", np.asarray(params["mm_lj_epsilon_scale"]))

# Training splits them back out before calling the network.
model_params, sig, eps = split_mm_lj_scale_params(params)
print("\nnetwork sees only:", sorted(model_params.keys()))

# Unit scales reproduce stock CGenFF exactly.
s, e = apply_mm_lj_scales(MASTER_SIGMAS, MASTER_EPSILONS, sig, eps)
print("\nunit scales identical to master tables:",
      bool(np.allclose(s, MASTER_SIGMAS) and np.allclose(e, MASTER_EPSILONS)))

## 6. Convince yourself the gradient reaches them

The whole approach rests on σ/ε being differentiable inside the hybrid energy. Do
not take that on faith — build a two-monomer system and differentiate.

We use a stand-in for the neural network that returns a constant, so every
gradient below is attributable to the MM term alone.

In [ ]:
from mmml.models.hybrid_energy import hybrid_forward

SWITCH_KW = dict(mm_switch_on=3.0, mm_switch_width=2.0,
                 ml_switch_width=1.0, complementary_handoff=False)

def constant_ml(params, *, atomic_numbers, positions, dst_idx, src_idx,
                batch_segments, batch_size, batch_mask, atom_mask):
    '''Placeholder for PhysNet: constant energy, zero forces.'''
    e = jnp.sum(atom_mask) * jnp.asarray(-1.0)
    return {"energy": e.reshape(batch_size, 1), "forces": jnp.zeros_like(positions)}

def dimer(separation_A=3.5, type_idx=(0, 1, 0, 1)):
    '''Two 2-atom monomers along x, close enough that intermolecular LJ is on.'''
    n = 4
    pos = jnp.array([[0., 0, 0], [1., 0, 0],
                     [separation_A, 0, 0], [separation_A + 1., 0, 0]], dtype=jnp.float32)
    i = jnp.arange(n)
    dst, src = [a.reshape(-1) for a in jnp.meshgrid(i, i, indexing="ij")]
    return {
        "R": pos, "Z": jnp.array([6, 1, 6, 1]),
        "mol_id": jnp.array([0, 0, 1, 1]).reshape(1, n),
        "cgenff_type_idx": jnp.array(type_idx).reshape(1, n),
        "cgenff_charge": jnp.zeros(n).reshape(1, n),   # zero charge -> pure LJ
        "atom_mask": jnp.ones(n, dtype=jnp.float32),
        "batch_mask": (dst != src).astype(jnp.float32),
        "dst_idx": dst, "src_idx": src,
        "batch_segments": jnp.zeros(n, dtype=jnp.int32),
    }

def e_mm(sigma_scale, epsilon_scale, batch):
    out = hybrid_forward(constant_ml, {"params": {}}, batch, 1,
                         MASTER_SIGMAS, MASTER_EPSILONS,
                         learn_mm_lj_scales=True,
                         mm_lj_sigma_scale=sigma_scale,
                         mm_lj_epsilon_scale=epsilon_scale, **SWITCH_KW)
    return jnp.asarray(out["e_mm"]).reshape(())

batch = dimer()
g_sig, g_eps = jax.grad(lambda s, e: e_mm(s, e, batch), argnums=(0, 1))(
    jnp.ones(2), jnp.ones(2))
print("dE/ds_sigma  :", np.asarray(g_sig))
print("dE/ds_epsilon:", np.asarray(g_eps))
print("\nfinite and non-zero ->",
      bool(np.all(np.isfinite(g_sig)) and np.any(np.abs(g_eps) > 0)))

## 7. A miniature fit — and the degeneracy that will bite you

Now plant a known answer and recover it. **Read this section carefully: it
contains the single most important practical lesson in the notebook.**

σ and ε are **mutually degenerate against an energy-only target**. A deeper well
with a slightly larger radius produces the same energy as a shallower well with a
smaller one. So if you fit both against energies alone, you can drive the loss to
zero and still recover the wrong parameters.

Below we fit ε with σ held fixed. Try setting `freeze=None` and watch the loss go
to zero while the recovered ε is wrong.

In [ ]:
import optax

def fit(target, batch, *, freeze="sigma", lr=3e-2, steps=400):
    def loss_fn(p):
        _, sig, eps = split_mm_lj_scale_params(p)
        if freeze == "sigma":
            sig = jax.lax.stop_gradient(sig)
        elif freeze == "epsilon":
            eps = jax.lax.stop_gradient(eps)
        return (e_mm(sig, eps, batch) - target) ** 2

    p = attach_mm_lj_scales({"params": {}}, 2)
    opt = optax.adam(lr); state = opt.init(p)
    loss0 = float(loss_fn(p))

    @jax.jit
    def step(p, s):
        loss, grads = jax.value_and_grad(loss_fn)(p)
        upd, s = opt.update(grads, s, p)
        return optax.apply_updates(p, upd), s, loss

    for _ in range(steps):
        p, state, _ = step(p, state)
    return p, loss0, float(loss_fn(p))

# One atom type only, so a single scalar target identifies s_eps[0] uniquely.
single = dimer(type_idx=(0, 0, 0, 0))
TRUTH = 1.6
target = e_mm(jnp.ones(2), jnp.array([TRUTH, 1.0]), single)

p, l0, l1 = fit(target, single, freeze="sigma")
_, sig_out, eps_out = split_mm_lj_scale_params(p)

print(f"loss {l0:.3e} -> {l1:.3e}")
print(f"recovered s_eps[0] = {float(np.asarray(eps_out)[0]):.4f}   (planted {TRUTH})")
print(f"type 1 absent from the system, stays at "
      f"{float(np.asarray(eps_out)[1]):.6f}  <- untouched types keep unit scale")
print(f"frozen sigma untouched: {np.asarray(sig_out)}")

Two results worth internalising:

- **The planted value comes back.** Convergence is not just "loss went down"; the
  parameter itself is recovered.
- **A type absent from the data keeps `s = 1.0` exactly.** This is what makes it
  safe to deploy a model trained on a dimer into a solvated box: solvent types the
  model never saw fall back to stock CGenFF rather than to garbage.

In real training the degeneracy is broken by **forces** (which constrain the shape
of the curve, not just its value) and by **many geometries at different
separations**. That is why a distance scan is a better fitting set than a pile of
equilibrium structures.

## 8. Running the real training

`learn_mm_lj_scales: true` requires `lr_solver: mic` and `mm_include_lj: true`.
Under `ewald` / `nvalchemiops_pme` the LJ term is removed from the energy
entirely, so there is nothing to differentiate and the flag is silently ineffective.

```bash
mmml physnet-train \
  --config examples/hybrid_mm_charges/train_fixed_lj_scales.yaml \
  --data artifacts/lj_scales_dcm/dcm_mp2_cgenff.npz \
  --ckpt-dir artifacts/lj_scales_dcm/ckpts \
  --tag hybrid_mm_fixed_lj_scales
```

Or submit the whole prep → train → MD pipeline:

```bash
mkdir -p artifacts/lj_scales_dcm
sbatch examples/hybrid_mm_charges/submit_lj_scales_scicore.sbatch
```

Check the log for `Learnable MM LJ scales enabled`. If it is missing, the flag was
overridden — almost always by an Ewald `lr_solver`.

## 9. Reading the result: `hybrid_mm.json`

Training writes the final (EMA) scale vectors next to the checkpoint. This
sidecar, not the checkpoint, is what MD reads.

In [ ]:
from mmml.models.mm_lj_scales import write_mm_lj_scales_into_hybrid_mm_json

# Stand-in for a real training output so this cell runs anywhere.
demo = Path("_demo_hybrid_mm.json")
write_mm_lj_scales_into_hybrid_mm_json(
    demo, type_names=TYPE_NAMES,
    sigma_scale=[1.0731, 0.9422], epsilon_scale=[1.6044, 0.5981])

payload = json.loads(demo.read_text())
print(json.dumps(payload, indent=2))

print("\n--- how to read it ---")
for name, s, e in zip(payload["cgenff_type_names"],
                      payload["mm_lj_sigma_scale"],
                      payload["mm_lj_epsilon_scale"]):
    print(f"  {name:8s} sigma x{s:.4f}  epsilon x{e:.4f}"
          f"{'   <- moved' if abs(s-1) > 1e-3 or abs(e-1) > 1e-3 else ''}")

A scale that is still exactly 1.0 means that type got no gradient — it never
appeared in your training data. That is expected for solvent types and is not a
bug. Scales far from 1.0 (say beyond 0.5–2.0) deserve suspicion: that is usually
compensation for something else being wrong, not a genuine parameter correction.

## 10. Deploying in MD — and the one trap that silently does nothing

MD does not use the master-table ordering. It maps scales onto CHARMM's atom-type
list (`param.get_atc()`), filling **1.0** for any type absent from training.

In [ ]:
from mmml.models.mm_lj_scales import scales_to_atc

# CHARMM's ordering differs from the master table and includes extra types.
atc = ["OG2D1", "HGA3", "CG331", "CLGA1"]
ep_scale, sig_scale = scales_to_atc(
    payload["cgenff_type_names"],
    payload["mm_lj_sigma_scale"],
    payload["mm_lj_epsilon_scale"],
    atc)

print(f"{'ATC type':10s} {'sig_scale':>10s} {'ep_scale':>10s}")
for name, s, e in zip(atc, sig_scale, ep_scale):
    tag = "" if abs(s-1) < 1e-9 and abs(e-1) < 1e-9 else "  <- trained"
    print(f"{name:10s} {s:10.4f} {e:10.4f}{tag}")

# This is exactly what mm_energy_forces does with them:
atc_epsilons = np.array([-0.1200, -0.0240, -0.0780, -0.3430])
atc_rmins    = np.array([ 1.7000,  1.3400,  2.0600,  1.9100])
print("\ndeployed eps :", -1 * np.abs(atc_epsilons) * ep_scale)
print("deployed rmin:", atc_rmins * sig_scale)

demo.unlink()  # tidy up

### The trap

`ep_scale`/`sig_scale` are consumed by the **JAX switched-MM pair loop**. That
loop is only active when JAX MM is on:

```python
do_mm = include_mm and not periodic_mode
```

So with `mm_nonbond_mode: periodic_external`, VDW is handed to CHARMM's IMAGE
code, which does not read `hybrid_mm.json`. **Your trained LJ would not be
applied.** This used to happen silently; MLpot now raises a `ValueError` if you
pass `--mm-lj-scales-file` in that mode, and warns on stderr if it auto-discovers
a sidecar it cannot use.

For a condensed-phase run with trained LJ, pin `jax_mic`:

```yaml
include_mm: true
mm_nonbond_mode: jax_mic     # the default; periodic_external cannot apply scales
setup: pbc_nvt
composition: "DCM:64"
box_size: 25.0
```

```bash
mmml md-system --config examples/hybrid_mm_charges/md_fixed_lj_scales.yaml \
               --only liquid_nvt --checkpoint <.../params.json>
```

The honest limitation: this uses **truncated-MIC electrostatics**, not Ewald.
Combining learned LJ with full Ewald in one production energy is not implemented —
that is [issue #139](https://github.com/EricBoittier/mmml/issues/139).

## 11. Checklist and troubleshooting

Before believing a result:

- [ ] Enriched NPZ has `cgenff_type_idx`, `cgenff_charge`, `mol_id`
- [ ] Training log says `Learnable MM LJ scales enabled`
- [ ] `hybrid_mm.json` exists and some scales moved off 1.0
- [ ] Scales are in a physically plausible band (roughly 0.5–2.0)
- [ ] MD log reports `Loaded MM LJ scales (N ATC types)`
- [ ] `mm_switch_on` / `mm_switch_width` / `ml_switch_width` **identical** to training
- [ ] Validated against a property not in the loss (density, RDF peak)

| Symptom | Cause |
|---|---|
| Scales all exactly 1.0 | `learn_mm_lj_scales` off, or `lr_solver` is ewald/PME (LJ removed from the energy) |
| `... but JAX MM is off` error | Deploying under `periodic_external` / `include_mm: false`, which cannot apply scales |
| `WARNING: ... carries trained MM LJ scales but JAX MM is off` | Sidecar auto-found but unusable — this run uses **stock** CGenFF LJ |
| Loss → 0 but parameters wrong | The σ/ε degeneracy of §7. Add forces and more geometries |
| Great fit, bad density under Ewald | Coulomb error absorbed into LJ. See §2 |
| ATC length mismatch | Sidecar type names disagree with CHARMM's; regenerate from the same CGenFF PRM |

Run the tests that pin all of the above:

```bash
uv run pytest tests/unit/test_mm_lj_scales.py \
              tests/unit/test_mm_lj_scales_learning.py -q
```